In [22]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from scipy import stats

In [23]:
df = pd.read_excel(r"C:\Users\ACER\Desktop\musteri_dataset.xlsx")
df.head(5)

,Müştəri ID,Müştəri Adı,Reklam Növü,Əmək Haqqı (AZN),Məhsul Alıb?,Saytda Vaxt (dəq),Yaş
0,MÜŞ-1001,Əli Həsənov,Instagram Ads,850,Yox,12.7,32
1,MÜŞ-1002,Nigar Quliyeva,Google Ads,3400,Yox,33.5,52
2,MÜŞ-1003,Rəşad Mammadov,Google Ads,2800,Hə,1.8,31
3,MÜŞ-1004,Aytən Əliyeva,Google Ads,2450,Hə,25.5,59
4,MÜŞ-1005,Kamran Hüseynov,Instagram Ads,2650,Hə,20.5,35


In [24]:
# 2. Dummies ilə Reklam Növünün avtomatik ayrılması (One-Hot Encoding)
# drop_first=True multikollinearlığın qarşısını alır və reklam növünün birini baza seçir
df_encoded = pd.get_dummies(df, columns=['Reklam Növü'], drop_first=True, dtype=int)
df_encoded.head(5)

,Müştəri ID,Müştəri Adı,Əmək Haqqı (AZN),Məhsul Alıb?,Saytda Vaxt (dəq),Yaş,Reklam Növü_Google Ads,Reklam Növü_Instagram Ads
0,MÜŞ-1001,Əli Həsənov,850,Yox,12.7,32,0,1
1,MÜŞ-1002,Nigar Quliyeva,3400,Yox,33.5,52,1,0
2,MÜŞ-1003,Rəşad Mammadov,2800,Hə,1.8,31,1,0
3,MÜŞ-1004,Aytən Əliyeva,2450,Hə,25.5,59,1,0
4,MÜŞ-1005,Kamran Hüseynov,2650,Hə,20.5,35,0,1


In [25]:
# 3. Asılı sütundakı 'Hə'-ni 1, 'Yox'-u 0 etmək
# 'Hə' görəndə 1, 'Yox' görəndə 0 ilə əvəzləyirik
df_encoded['Məhsul Alıb?'] = df_encoded['Məhsul Alıb?'].replace({'Hə': 1, 'Yox': 0}).astype(int)
df_encoded.head(5)

,Müştəri ID,Müştəri Adı,Əmək Haqqı (AZN),Məhsul Alıb?,Saytda Vaxt (dəq),Yaş,Reklam Növü_Google Ads,Reklam Növü_Instagram Ads
0,MÜŞ-1001,Əli Həsənov,850,0,12.7,32,0,1
1,MÜŞ-1002,Nigar Quliyeva,3400,0,33.5,52,1,0
2,MÜŞ-1003,Rəşad Mammadov,2800,1,1.8,31,1,0
3,MÜŞ-1004,Aytən Əliyeva,2450,1,25.5,59,1,0
4,MÜŞ-1005,Kamran Hüseynov,2650,1,20.5,35,0,1


In [27]:


# 1. Excel faylını oxuyuruq (əgər yuxarıdakı kodla eyni fayldırsa, yenidən oxumağa ehtiyac yoxdur)
# df = pd.read_excel("dataset.xlsx")

# 2. Kvartillərin (Q1, Q2, Q3) hesablanması
Q1 = df['Əmək Haqqı (AZN)'].quantile(0.25)
Q2 = df['Əmək Haqqı (AZN)'].quantile(0.50)  # Bu həm də Mediandır
Q3 = df['Əmək Haqqı (AZN)'].quantile(0.75)

# 3. IQR-ın hesablanması
IQR = Q3 - Q1

# 4. Outlier-lər üçün alt və üst sərhədlərin təyin edilməsi
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Hesablanmış dəyərləri ekrana yazdırırıq
print("--- Kvartillər və IQR Hesabatı ---")
print(f"Q1 (25-ci faiz dərəcəsi): {Q1} AZN")
print(f"Q2 (Median - 50-ci faiz dərəcəsi): {Q2} AZN")
print(f"Q3 (75-ci faiz dərəcəsi): {Q3} AZN")
print(f"IQR (Kvartillərarası Fərq): {IQR} AZN")
print(f"Aşağı Sərhəd (Lower Bound): {lower_bound} AZN")
print(f"Yuxarı Sərhəd (Upper Bound): {upper_bound} AZN")
print("-" * 33)

# 5. Outlier-lərin (kənarlaşmaların) yoxlanılması
# Sərhədlərdən kənarda qalan dataları süzürük
outliers = df[(df['Əmək Haqqı (AZN)'] < lower_bound) | (df['Əmək Haqqı (AZN)'] > upper_bound)]

# 6. Nəticənin ekrana çıxarılması
if not outliers.empty:
    print(f"BƏLİ, datada {len(outliers)} dənə outlier (kənarlaşma) tapıldı:")
    # Outlier olan müştərilərin ID-si, Adı və Əmək haqqını göstəririk
    print(outliers[['Müştəri ID', 'Müştəri Adı', 'Əmək Haqqı (AZN)']])
else:
    print("XEYR, datada heç bir outlier (kənarlaşma) tapılmadı. Bütün əməkhaqqı dəyərləri normal sərhədlər daxilindədir.")

--- Kvartillər və IQR Hesabatı ---
Q1 (25-ci faiz dərəcəsi): 975.0 AZN
Q2 (Median - 50-ci faiz dərəcəsi): 1950.0 AZN
Q3 (75-ci faiz dərəcəsi): 2800.0 AZN
IQR (Kvartillərarası Fərq): 1825.0 AZN
Aşağı Sərhəd (Lower Bound): -1762.5 AZN
Yuxarı Sərhəd (Upper Bound): 5537.5 AZN
---------------------------------
XEYR, datada heç bir outlier (kənarlaşma) tapılmadı. Bütün əməkhaqqı dəyərləri normal sərhədlər daxilindədir.


In [28]:
# Məhsul alanların orta əmək haqqı ilə məhsul almayanların orta 
# əmək haqqı arasında statistik olaraq ciddi bir fərq varmı?

In [33]:


alanlar = df[df['Məhsul Alıb?'] == 'Hə']['Əmək Haqqı (AZN)']
almayanlar = df[df['Məhsul Alıb?'] == 'Yox']['Əmək Haqqı (AZN)']

t_stat, p_val = stats.ttest_ind(alanlar, almayanlar)

print(alanlar,end="\n\n")
print(almayanlar,end="\n\n")
print(f"t-statistikası: {t_stat}, p-value: {p_val}")

2      2800
3      2450
4      2650
5      1050
8       900
       ... 
190    2950
191    1500
195     900
197    2250
198    3450
Name: Əmək Haqqı (AZN), Length: 123, dtype: int64

0       850
1      3400
6      1800
7      2850
10     3300
       ... 
185    2050
192    2400
193    3300
194     800
196    3300
Name: Əmək Haqqı (AZN), Length: 76, dtype: int64

t-statistikası: -0.25527736722805555, p-value: 0.7987751981503983


In [ ]:
# Müştərinin yaşı artdıqca onun əmək haqqı da artırmı? Bu iki dəyişən arasında xətti bir əlaqə varmı?

In [34]:
korrelyasiya = df['Yaş'].corr(df['Əmək Haqqı (AZN)'])
print(f"Yaş və Əmək Haqqı arası korrelyasiya: {korrelyasiya}")

Yaş və Əmək Haqqı arası korrelyasiya: 0.0346769768114452
